In [2]:
# 1-dataset model (HTCas9)

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc5():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc5():
    with open('RfxCas13d_validation_expression_level_combined.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc5 = load_branch1_data_RfxCas13d_mmc5()
    X1_RfxCas13d_mmc5   = np.asarray(X1_RfxCas13d_mmc5)
    X1 = np.concatenate([X1_RfxCas13d_mmc5], axis=0) 

    rates_RfxCas13d_mmc5 = load_reaction_rates_RfxCas13d_mmc5()
    rates_RfxCas13d_mmc5   = np.asarray(rates_RfxCas13d_mmc5)
    rates = np.concatenate([rates_RfxCas13d_mmc5], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc5 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc5 = np.arange(len(rates_RfxCas13d_mmc5))
    selected_indices_RfxCas13d_mmc5 = np.random.choice(len(full_indices_RfxCas13d_mmc5), size=len(full_indices_RfxCas13d_mmc5), replace=False)
    unseen_indices_RfxCas13d_mmc5 = np.setdiff1d(full_indices_RfxCas13d_mmc5, selected_indices_RfxCas13d_mmc5)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc5)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc5)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc5, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_1_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_1_dataset_trained_CNN1.pt: Spearman = 0.185904143596706
../../../../CNN1_only_models/trial_1_best_1_dataset_trained_CNN1.pt: Spearman = 0.20365321239473913
../../../../CNN1_only_models/trial_2_best_1_dataset_trained_CNN1.pt: Spearman = 0.10785080125211773
../../../../CNN1_only_models/trial_3_best_1_dataset_trained_CNN1.pt: Spearman = 0.20890328717257411
../../../../CNN1_only_models/trial_4_best_1_dataset_trained_CNN1.pt: Spearman = 0.17185345553591752
../../../../CNN1_only_models/trial_5_best_1_dataset_trained_CNN1.pt: Spearman = 0.2190580257823796
../../../../CNN1_only_models/trial_6_best_1_dataset_trained_CNN1.pt: Spearman = 0.17912966417703458
../../../../CNN1_only_models/trial_7_best_1_dataset_trained_CNN1.pt: Spearman = 0.18109133135260977
../../../../CNN1_only_models/trial_8_best_1_dataset_trained_CNN1.pt: Spearman = 0.13114853011232877
../../../../CNN1_only_models/trial_9_best_1_dataset_trained_CNN1.pt: Spearman = 

In [4]:
# 2-dataset model (HTCas9+HT11)

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc5():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc5():
    with open('RfxCas13d_validation_expression_level_combined.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc5 = load_branch1_data_RfxCas13d_mmc5()
    X1_RfxCas13d_mmc5   = np.asarray(X1_RfxCas13d_mmc5)
    X1 = np.concatenate([X1_RfxCas13d_mmc5], axis=0) 

    rates_RfxCas13d_mmc5 = load_reaction_rates_RfxCas13d_mmc5()
    rates_RfxCas13d_mmc5   = np.asarray(rates_RfxCas13d_mmc5)
    rates = np.concatenate([rates_RfxCas13d_mmc5], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc5 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc5 = np.arange(len(rates_RfxCas13d_mmc5))
    selected_indices_RfxCas13d_mmc5 = np.random.choice(len(full_indices_RfxCas13d_mmc5), size=len(full_indices_RfxCas13d_mmc5), replace=False)
    unseen_indices_RfxCas13d_mmc5 = np.setdiff1d(full_indices_RfxCas13d_mmc5, selected_indices_RfxCas13d_mmc5)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc5)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc5)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc5, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_2_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_2_dataset_trained_CNN1.pt: Spearman = 0.28247572229547524
../../../../CNN1_only_models/trial_1_best_2_dataset_trained_CNN1.pt: Spearman = 0.3054155220103033
../../../../CNN1_only_models/trial_2_best_2_dataset_trained_CNN1.pt: Spearman = 0.26097568092466433
../../../../CNN1_only_models/trial_3_best_2_dataset_trained_CNN1.pt: Spearman = 0.24319379615405737
../../../../CNN1_only_models/trial_4_best_2_dataset_trained_CNN1.pt: Spearman = 0.2455056643965263
../../../../CNN1_only_models/trial_5_best_2_dataset_trained_CNN1.pt: Spearman = 0.3311617789276863
../../../../CNN1_only_models/trial_6_best_2_dataset_trained_CNN1.pt: Spearman = 0.31924577068296345
../../../../CNN1_only_models/trial_7_best_2_dataset_trained_CNN1.pt: Spearman = 0.2553541566970631
../../../../CNN1_only_models/trial_8_best_2_dataset_trained_CNN1.pt: Spearman = 0.23351223987484948
../../../../CNN1_only_models/trial_9_best_2_dataset_trained_CNN1.pt: Spearman = 0

In [6]:
# 4-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5))

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc5():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc5():
    with open('RfxCas13d_validation_expression_level_combined.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc5 = load_branch1_data_RfxCas13d_mmc5()
    X1_RfxCas13d_mmc5   = np.asarray(X1_RfxCas13d_mmc5)
    X1 = np.concatenate([X1_RfxCas13d_mmc5], axis=0) 

    rates_RfxCas13d_mmc5 = load_reaction_rates_RfxCas13d_mmc5()
    rates_RfxCas13d_mmc5   = np.asarray(rates_RfxCas13d_mmc5)
    rates = np.concatenate([rates_RfxCas13d_mmc5], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc5 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc5 = np.arange(len(rates_RfxCas13d_mmc5))
    selected_indices_RfxCas13d_mmc5 = np.random.choice(len(full_indices_RfxCas13d_mmc5), size=len(full_indices_RfxCas13d_mmc5), replace=False)
    unseen_indices_RfxCas13d_mmc5 = np.setdiff1d(full_indices_RfxCas13d_mmc5, selected_indices_RfxCas13d_mmc5)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc5)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc5)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc5, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_4_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_4_dataset_trained_CNN1.pt: Spearman = 0.6827280831857282
../../../../CNN1_only_models/trial_1_best_4_dataset_trained_CNN1.pt: Spearman = 0.6744189056757611
../../../../CNN1_only_models/trial_2_best_4_dataset_trained_CNN1.pt: Spearman = 0.7562279725481454
../../../../CNN1_only_models/trial_3_best_4_dataset_trained_CNN1.pt: Spearman = 0.7232521662373163
../../../../CNN1_only_models/trial_4_best_4_dataset_trained_CNN1.pt: Spearman = 0.6891662312532115
../../../../CNN1_only_models/trial_5_best_4_dataset_trained_CNN1.pt: Spearman = 0.5275867303006888
../../../../CNN1_only_models/trial_6_best_4_dataset_trained_CNN1.pt: Spearman = 0.48568209591309114
../../../../CNN1_only_models/trial_7_best_4_dataset_trained_CNN1.pt: Spearman = 0.7084667771741169
../../../../CNN1_only_models/trial_8_best_4_dataset_trained_CNN1.pt: Spearman = 0.7574877653176126
../../../../CNN1_only_models/trial_9_best_4_dataset_trained_CNN1.pt: Spearman = 0.732

In [8]:
# 5-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq)

In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc5():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc5():
    with open('RfxCas13d_validation_expression_level_combined.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc5 = load_branch1_data_RfxCas13d_mmc5()
    X1_RfxCas13d_mmc5   = np.asarray(X1_RfxCas13d_mmc5)
    X1 = np.concatenate([X1_RfxCas13d_mmc5], axis=0) 

    rates_RfxCas13d_mmc5 = load_reaction_rates_RfxCas13d_mmc5()
    rates_RfxCas13d_mmc5   = np.asarray(rates_RfxCas13d_mmc5)
    rates = np.concatenate([rates_RfxCas13d_mmc5], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc5 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc5 = np.arange(len(rates_RfxCas13d_mmc5))
    selected_indices_RfxCas13d_mmc5 = np.random.choice(len(full_indices_RfxCas13d_mmc5), size=len(full_indices_RfxCas13d_mmc5), replace=False)
    unseen_indices_RfxCas13d_mmc5 = np.setdiff1d(full_indices_RfxCas13d_mmc5, selected_indices_RfxCas13d_mmc5)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc5)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc5)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc5, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_5_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_5_dataset_trained_CNN1.pt: Spearman = 0.7615318522622214
../../../../CNN1_only_models/trial_1_best_5_dataset_trained_CNN1.pt: Spearman = 0.7759223522040121
../../../../CNN1_only_models/trial_2_best_5_dataset_trained_CNN1.pt: Spearman = 0.7307221307306606
../../../../CNN1_only_models/trial_3_best_5_dataset_trained_CNN1.pt: Spearman = 0.5840352063519395
../../../../CNN1_only_models/trial_4_best_5_dataset_trained_CNN1.pt: Spearman = 0.7437568802568021
../../../../CNN1_only_models/trial_5_best_5_dataset_trained_CNN1.pt: Spearman = 0.7571615589488244
../../../../CNN1_only_models/trial_6_best_5_dataset_trained_CNN1.pt: Spearman = 0.6022224402143547
../../../../CNN1_only_models/trial_7_best_5_dataset_trained_CNN1.pt: Spearman = 0.6082575313007008
../../../../CNN1_only_models/trial_8_best_5_dataset_trained_CNN1.pt: Spearman = 0.4207550357170854
../../../../CNN1_only_models/trial_9_best_5_dataset_trained_CNN1.pt: Spearman = 0.4957

In [10]:
# 6-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER)

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc5():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc5():
    with open('RfxCas13d_validation_expression_level_combined.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc5 = load_branch1_data_RfxCas13d_mmc5()
    X1_RfxCas13d_mmc5   = np.asarray(X1_RfxCas13d_mmc5)
    X1 = np.concatenate([X1_RfxCas13d_mmc5], axis=0) 

    rates_RfxCas13d_mmc5 = load_reaction_rates_RfxCas13d_mmc5()
    rates_RfxCas13d_mmc5   = np.asarray(rates_RfxCas13d_mmc5)
    rates = np.concatenate([rates_RfxCas13d_mmc5], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc5 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc5 = np.arange(len(rates_RfxCas13d_mmc5))
    selected_indices_RfxCas13d_mmc5 = np.random.choice(len(full_indices_RfxCas13d_mmc5), size=len(full_indices_RfxCas13d_mmc5), replace=False)
    unseen_indices_RfxCas13d_mmc5 = np.setdiff1d(full_indices_RfxCas13d_mmc5, selected_indices_RfxCas13d_mmc5)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc5)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc5)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc5, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_6_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_6_dataset_trained_CNN1.pt: Spearman = 0.5085674490253085
../../../../CNN1_only_models/trial_1_best_6_dataset_trained_CNN1.pt: Spearman = 0.48182301909318864
../../../../CNN1_only_models/trial_2_best_6_dataset_trained_CNN1.pt: Spearman = 0.5045475767350145
../../../../CNN1_only_models/trial_3_best_6_dataset_trained_CNN1.pt: Spearman = 0.7021455942310306
../../../../CNN1_only_models/trial_4_best_6_dataset_trained_CNN1.pt: Spearman = 0.6928043142356801
../../../../CNN1_only_models/trial_5_best_6_dataset_trained_CNN1.pt: Spearman = 0.6759866910812234
../../../../CNN1_only_models/trial_6_best_6_dataset_trained_CNN1.pt: Spearman = 0.6709101184683368
../../../../CNN1_only_models/trial_7_best_6_dataset_trained_CNN1.pt: Spearman = 0.7043203649154004
../../../../CNN1_only_models/trial_8_best_6_dataset_trained_CNN1.pt: Spearman = 0.71662693828514
../../../../CNN1_only_models/trial_9_best_6_dataset_trained_CNN1.pt: Spearman = 0.74841

In [12]:
# 7-dataset model (HTCas9+HT11+RfxCas13d(mmc4+mmc5)+CHANGE_seq+TIGER+iMeta)

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data_RfxCas13d_mmc5():
    """
    Loads (60,4) data from "Feature_CNN1_reduced.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1_reduced.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 60, 4)
    return X_branch1

def load_reaction_rates_RfxCas13d_mmc5():
    with open('RfxCas13d_validation_expression_level_combined.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)


# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))


########################################
# 3. CNN1 Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (60,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*30, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,30)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,30)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x


########################################
# 4. Final Fusion Model
########################################

class CNN1OnlyModel(nn.Module):
    def __init__(self, filters1, kernel_size1, dense_units1, final_fc_dim, dropout_rate=0.0):
        super().__init__()
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(dense_units1, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    def forward(self, x1):
        feat1 = self.cnn_branch1(x1)
        merged = F.dropout(feat1, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))
        return self.out(x).view(-1)
        
########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, X1, reaction_rates):
        super().__init__()
        self.X1 = X1
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)
        y_val = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return x1, y_val


def hybrid_collate(batch):
    x1_list, y_list = zip(*batch)
    x1 = torch.stack(x1_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return x1, y


########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    
    # List all model files you want to evaluate
    model_paths = [
        "../../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt",
        "../../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt"
    ]
    
    # final hold-out evaluation
    # load data

    X1_RfxCas13d_mmc5 = load_branch1_data_RfxCas13d_mmc5()
    X1_RfxCas13d_mmc5   = np.asarray(X1_RfxCas13d_mmc5)
    X1 = np.concatenate([X1_RfxCas13d_mmc5], axis=0) 

    rates_RfxCas13d_mmc5 = load_reaction_rates_RfxCas13d_mmc5()
    rates_RfxCas13d_mmc5   = np.asarray(rates_RfxCas13d_mmc5)
    rates = np.concatenate([rates_RfxCas13d_mmc5], axis=0) 

    hybrid_dataset = HybridDataset(X1, rates)

    # Get unseen RfxCas13d_mmc5 dataset
    np.random.seed(42)
    full_indices_RfxCas13d_mmc5 = np.arange(len(rates_RfxCas13d_mmc5))
    selected_indices_RfxCas13d_mmc5 = np.random.choice(len(full_indices_RfxCas13d_mmc5), size=len(full_indices_RfxCas13d_mmc5), replace=False)
    unseen_indices_RfxCas13d_mmc5 = np.setdiff1d(full_indices_RfxCas13d_mmc5, selected_indices_RfxCas13d_mmc5)

    from torch.utils.data import ConcatDataset, DataLoader

    # Evaluate on 10 subsamples from unseen
    unseen_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, unseen_indices_RfxCas13d_mmc5)
    
    from torch.utils.data import ConcatDataset, DataLoader

    selected_set_RfxCas13d_mmc5 = Subset(hybrid_dataset, selected_indices_RfxCas13d_mmc5)
    trial_loader = DataLoader(selected_set_RfxCas13d_mmc5, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    
    spearman_results = []

    for model_path in model_paths:
        model = torch.load(model_path, weights_only=False)
        model.eval()

        preds_trial = []
        labels_trial = []

        with torch.no_grad():
            for xx1_b, yy_b in trial_loader:
                p = model(xx1_b)
                preds_trial.append(p.item())
                labels_trial.append(yy_b.item())

        sp_corr, _ = spearmanr(labels_trial, preds_trial)

        print(f"{model_path}: Spearman = {sp_corr}")
        spearman_results.append((model_path, sp_corr))

    with open("Spearman_all_models_7_dataset_CNN1.txt", "w") as f:
        for _, sp_corr in spearman_results:
            f.write(f"{sp_corr}\n")
    

if __name__ == "__main__":
    main_pipeline() 


Using PyTorch and PyG.
../../../../CNN1_only_models/trial_0_best_7_dataset_trained_CNN1.pt: Spearman = 0.49228726504609277
../../../../CNN1_only_models/trial_1_best_7_dataset_trained_CNN1.pt: Spearman = 0.5714402439397553
../../../../CNN1_only_models/trial_2_best_7_dataset_trained_CNN1.pt: Spearman = 0.7300762142575415
../../../../CNN1_only_models/trial_3_best_7_dataset_trained_CNN1.pt: Spearman = 0.5976296111747686
../../../../CNN1_only_models/trial_4_best_7_dataset_trained_CNN1.pt: Spearman = 0.6615462225441465
../../../../CNN1_only_models/trial_5_best_7_dataset_trained_CNN1.pt: Spearman = 0.7303941827605857
../../../../CNN1_only_models/trial_6_best_7_dataset_trained_CNN1.pt: Spearman = 0.72162804567426
../../../../CNN1_only_models/trial_7_best_7_dataset_trained_CNN1.pt: Spearman = 0.7664917494471055
../../../../CNN1_only_models/trial_8_best_7_dataset_trained_CNN1.pt: Spearman = 0.6888171402079764
../../../../CNN1_only_models/trial_9_best_7_dataset_trained_CNN1.pt: Spearman = 0.66711